In [1]:
import pandas as pd
import sqlite3

## 1. Connection

In [2]:
db = sqlite3.connect("../data/checking-logs.sqlite")

## 2. Join in new table

- сначала посмотрим на таблички

In [3]:
pd.io.sql.read_sql("PRAGMA table_info(pageviews);", db)


,cid,name,type,notnull,dflt_value,pk
0,0,index,INTEGER,0,None,0
1,1,uid,TEXT,0,None,0
2,2,datetime,TIMESTAMP,0,None,0


In [4]:
pd.io.sql.read_sql("PRAGMA table_info(checker);", db)


,cid,name,type,notnull,dflt_value,pk
0,0,index,INTEGER,0,None,0
1,1,status,TEXT,0,None,0
2,2,success,INTEGER,0,None,0
3,3,timestamp,TIMESTAMP,0,None,0
4,4,numTrials,INTEGER,0,None,0
5,5,labname,TEXT,0,None,0
6,6,uid,TEXT,0,None,0


- Поэтапно расширяю свой *query* изучая как он работает

In [5]:
query = """
SELECT 
    l.uid, 
    l.labname, 
    l.timestamp AS first_commit_ts,
    r.datetime AS first_view_ts
FROM checker AS l JOIN pageviews AS r 
    ON r.uid = l.uid
"""
df = pd.io.sql.read_sql(query, db)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160367 entries, 0 to 160366
Data columns (total 4 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   uid              160367 non-null  object
 1   labname          159244 non-null  object
 2   first_commit_ts  160367 non-null  object
 3   first_view_ts    160367 non-null  object
dtypes: object(4)
memory usage: 4.9+ MB


- Короче все почти так же как в предыдущке.. но добавляем LEFT JOIN. 

- Попытка создать таблицу в БД не увенчалась успехом - создадим просто дф  
P.S. каким-то чудом создалась..  
Там еще дропаем ее, чтобы перезаписывалась



In [6]:
query = """
CREATE TABLE datamart AS
SELECT 
    l.uid, 
    l.labname, 
    datetime(MIN(l.timestamp)) AS first_commit_ts,
    datetime(MIN(r.datetime)) AS first_view_ts
FROM checker AS l LEFT JOIN pageviews AS r 
    ON r.uid = l.uid
    WHERE l.status = 'ready'
        AND l.numTrials = 1
        AND l.labname IN ('laba04','laba04s','laba05','laba06','laba06s','project1')
        AND l.uid LIKE 'user_%'
GROUP BY l.uid, l.labname;
"""
db.execute("DROP TABLE IF EXISTS datamart;")
db.execute(query)
pd.io.sql.read_sql("PRAGMA table_info(datamart);", db)



,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,,0,None,0
3,3,first_view_ts,,0,None,0


- Записываем в датафрейм

In [7]:
datamart = pd.io.sql.read_sql("SELECT* from datamart", db, parse_dates=["first_view_ts", "first_commit_ts"])
datamart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              140 non-null    object        
 1   labname          140 non-null    object        
 2   first_commit_ts  140 non-null    datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 4.5+ KB


In [8]:
# query = """
# SELECT 
#     l.uid, 
#     l.labname, 
#     datetime(MIN(l.timestamp)) AS first_commit_ts,
#     datetime(MIN(r.datetime)) AS first_view_ts
# FROM checker AS l LEFT JOIN pageviews AS r 
#     ON r.uid = l.uid
#     WHERE l.status = 'ready'
#         AND l.numTrials = 1
#         AND l.labname IN ('laba04','laba04s','laba05','laba06','laba06s','project1')
#         AND l.uid LIKE 'user_%'
# GROUP BY l.uid, l.labname;
# """
# datamart = pd.read_sql(query, db, parse_dates=["first_view_ts", "first_commit_ts"])
# datamart.dtypes

## 3. Test n Control

- Создание двух новых датасетов

In [9]:
test    = datamart[datamart["first_view_ts"].notna()].copy()
control = datamart[datamart["first_view_ts"].isna()].copy()
test.info()
control

<class 'pandas.core.frame.DataFrame'>
Index: 59 entries, 0 to 114
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              59 non-null     object        
 1   labname          59 non-null     object        
 2   first_commit_ts  59 non-null     datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 2.3+ KB


,uid,labname,first_commit_ts,first_view_ts
12,user_11,laba05,2020-05-03 21:06:55,NaT
13,user_11,project1,2020-05-03 23:45:33,NaT
14,user_12,laba04,2020-04-18 17:07:51,NaT
15,user_12,laba04s,2020-04-26 15:42:38,NaT
16,user_12,laba05,2020-05-03 08:39:25,NaT
...,...,...,...,...
135,user_8,laba04s,2020-04-19 10:22:35,NaT
136,user_8,laba05,2020-05-02 13:28:07,NaT
137,user_8,laba06,2020-05-16 17:56:15,NaT
138,user_8,laba06s,2020-05-16 20:01:07,NaT


- заполнение второго (там столбец был пустым, как это видно)

In [10]:
control["first_view_ts"] = test["first_view_ts"].mean()
control.info()

<class 'pandas.core.frame.DataFrame'>
Index: 81 entries, 12 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              81 non-null     object        
 1   labname          81 non-null     object        
 2   first_commit_ts  81 non-null     datetime64[ns]
 3   first_view_ts    81 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 3.2+ KB


- сохраненеие их в базу данных

In [11]:
test.to_sql("test", db, if_exists="replace", index=False)
control.to_sql("control", db, if_exists="replace", index=False)

81

In [12]:
pd.io.sql.read_sql("PRAGMA table_info(test);", db)


,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


In [13]:
test = pd.io.sql.read_sql("SELECT* FROM test", db)
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   uid              59 non-null     object
 1   labname          59 non-null     object
 2   first_commit_ts  59 non-null     object
 3   first_view_ts    59 non-null     object
dtypes: object(4)
memory usage: 2.0+ KB


In [14]:
pd.io.sql.read_sql("PRAGMA table_info(control);", db)


,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


## 4. Close connection

In [15]:
db.close()